# v4 — Isotropic pooled-grid UNet detector (3D heatmap regression, self-contained)

Same detector family as v1, but the network sees an **isotropic 64³ grid**: XY is **max-pooled by 4** so the
voxel is 1.625 µm on all three axes (z untouched = 1.625; xy 0.40625 × 4 = 1.625). z is NOT pooled. This is
the single architectural change the reference LB-0.843 notebook used over our full-res anisotropic v1 (0.827).

**Single-variable vs v1 (deliberate):** same `base=16`, InstanceNorm, masked-MSE ignore loss, cosine LR,
60-epoch schedule. The ONLY change is the isotropic grid + symmetric strides. Everything downstream (peak
coords mapped back to ORIGINAL voxel space, full-res centroid refine, NN linking, edge-Jaccard scorer) is
identical to v1, so the eval cell's edge-Jaccard is directly comparable to **v1's 0.808** under the same NN
linker — the delta is purely "isotropic geometry".

**Pipeline:** pool XY→64³ → UNet heatmap → peaks (on 64³) → map xy back ×4 → **refine on full-res** (recovers
sub-pool xy precision) → NN link → edge-Jaccard. GPU (T4) ON, Internet ON, competition data as input, Run All.

**After training:** compare eval edge-J to v1's 0.808. If up, fuse this detector with v2 linking in
`submit.ipynb` and LB-test (HM_THR needs re-calibration — the pooled grid changes peak density).

In [ ]:
# --- deps (internet ON during training) ---
import subprocess, sys
try:
    import zarr; assert int(zarr.__version__.split('.')[0]) >= 3
except Exception:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'zarr>=3']); import zarr

import os, glob, time
from collections import defaultdict
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, RandomSampler
import scipy.ndimage as ndi
from scipy.ndimage import binary_dilation
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import cdist
from skimage.feature import peak_local_max
print('zarr', zarr.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# ================= CONFIG (v4: isotropic pooled-grid detector) =================
INPUT     = '/kaggle/input/competitions/biohub-cell-tracking-during-development'
TRAIN_DIR = os.path.join(INPUT, 'train')
VOXEL     = np.array([1.625, 0.40625, 0.40625])   # (z,y,x) um/voxel of the ORIGINAL grid (anisotropic)

# ---- v4 isotropic representation (the ONLY change vs v1) ----
# Max-pool XY by 4 -> the network sees an ISOTROPIC 64^3 grid (1.625 um on all axes: z untouched,
# xy 0.40625*4=1.625). z is NOT pooled. Downstream stays in ORIGINAL voxel space so edge-J is v1-comparable.
POOL_XY   = 4                    # XY downsample factor -> 256/4 = 64 -> isotropic 64^3
POOL_MODE = 'max'                # 'max' preserves bright centroids (chosen); 'mean' = area resample
VOXEL_ISO = np.array([VOXEL[0], VOXEL[1]*POOL_XY, VOXEL[2]*POOL_XY])  # (1.625,1.625,1.625) pooled um/voxel

# target / detection (SIGMA is ISOTROPIC in pooled voxels; 1 pooled voxel = 1.625 um)
SIGMA      = (1.0, 1.0, 1.0)     # isotropic Gaussian on the pooled grid
NORM_PCT   = (50.0, 99.8)
FG_THR     = 0.15                # bright-but-unlabeled -> ignore in loss
POS_WEIGHT = 10.0
MATCH_UM   = 7.0                 # scorer node-match radius (official metric; ORIGINAL um)
GATE_UM    = 8.0                 # NN linking gate (ORIGINAL um)

# model / training
BASE       = 16                  # single-variable: same as v1 (NOT the reference's base24)
BATCH      = 8                   # 64^3 is small -> larger batch than v1's 2
LR         = 2e-4
EPOCHS     = 60
STEPS      = 300                 # steps per epoch
N_VAL      = 20                  # last N samples held out
NUM_WORKERS= 2
STRIDES    = ((2,2,2),(2,2,2),(2,2,2),(2,2,2))   # SYMMETRIC (isotropic): 64->32->16->8->4

# inference (peaks on the 64^3 grid -> mapped back to ORIGINAL coords -> refined on full-res)
HM_THR          = 0.3            # heatmap peak threshold (NEEDS LB re-calibration: pooled grid changed density)
MIN_DIST_POOLED = 1              # peak separation in POOLED voxels (1 = 1.625 um; NOT a full-res min_distance)
REFINE          = True           # intensity-weighted CoM on the ORIGINAL full-res volume -> recover xy precision
REFINE_WIN      = (1, 5, 5)      # CoM window (z small: z is native full-res; xy ~ +/-2 um around the x4 map)

# LR schedule (cosine, resume-safe) -- same as the converged v1 run
COSINE_LR  = True
ETA_MIN    = LR * 0.02

# ---- EVAL-ONLY mode: skip training, load uploaded weights, just score val edge-Jaccard ----
EVAL_ONLY    = False   # True = don't train; load weights from an input dataset and score.
EVAL_WEIGHTS = None    # explicit path to a .pt under /kaggle/input; None = auto-find (unet_iso_*.pt)
EVAL_N_VAL   = 5       # held-out val samples to score in the eval cell (raise toward N_VAL for a firmer number)

# early stopping + time-budget safe-exit + resume
EARLY_STOP_PATIENCE = 8
TIME_BUDGET_SEC     = 8.5 * 3600
RESERVE_SEC         = 900
NB_START            = time.time()

# checkpoints (in /kaggle/working). DISTINCT names from v1 (different arch -> weights are NOT interchangeable).
OUT_WEIGHTS = '/kaggle/working/unet_iso_heatmap.pt'   # BEST model weights (state_dict)
CKPT_LATEST = '/kaggle/working/unet_iso_latest.pt'    # FULL checkpoint (model+opt+scaler+sched+epoch+best+history)
HISTORY_CSV = '/kaggle/working/history_iso.csv'       # per-epoch trend (epoch,train,val,best,lr,sec)
device = 'cuda' if torch.cuda.is_available() else 'cpu'


In [ ]:
# ================= IO =================
def open_image(zarr_path):
    g = zarr.open(zarr_path, mode='r')
    if hasattr(g, 'shape'):
        return g
    best, bestsize = None, -1
    def walk(node):
        nonlocal best, bestsize
        try:
            for k in node.keys():
                sub = node[k]
                if hasattr(sub, 'shape'):
                    s = int(np.prod(sub.shape))
                    if s > bestsize: best, bestsize = sub, s
                else: walk(sub)
        except Exception: pass
    walk(g); return best

def load_geff(geff_path):
    g = zarr.open(geff_path, mode='r')
    node_ids = np.asarray(g['nodes/ids'][:])
    props = {k: np.asarray(g[f'nodes/props/{k}/values'][:]) for k in g['nodes/props'].keys()}
    try: edges = np.asarray(g['edges/ids'][:])
    except Exception: edges = np.zeros((0, 2), dtype=np.uint64)
    return node_ids, props, edges

def points_by_frame(geff_path):
    _, props, _ = load_geff(geff_path)
    frames = defaultdict(list)
    for i in range(len(props['t'])):
        frames[int(props['t'][i])].append((int(props['z'][i]), int(props['y'][i]), int(props['x'][i])))
    return dict(frames)

def gt_nodes_edges(geff_path):
    node_ids, props, edges = load_geff(geff_path)
    nodes = [(int(node_ids[i]), int(props['t'][i]), int(props['z'][i]), int(props['y'][i]), int(props['x'][i]))
             for i in range(len(node_ids))]
    return nodes, [(int(s), int(t)) for s, t in edges]

def build_sample_list(train_dir):
    out = []
    for gp in sorted(glob.glob(os.path.join(train_dir, '*.geff'))):
        zp = gp[:-5] + '.zarr'
        if os.path.exists(zp): out.append((zp, gp))
    return out

def pool_xy(vol, factor=POOL_XY, mode=POOL_MODE):
    """Downsample a (Z,Y,X) volume in Y,X by `factor` (z untouched) -> isotropic grid.
    Block reduce: 256/4 = 64. 'max' keeps bright centroids; 'mean' = area resample."""
    Z, Y, X = vol.shape
    Yc, Xc = (Y // factor) * factor, (X // factor) * factor
    v = vol[:, :Yc, :Xc].astype(np.float32).reshape(Z, Yc // factor, factor, Xc // factor, factor)
    return v.max(axis=(2, 4)) if mode == 'max' else v.mean(axis=(2, 4))

In [ ]:
# ================= TARGETS: Gaussian heatmap + ignore mask (grid-agnostic; called on POOLED vols) =================
def _normalize(vol, norm_pct=NORM_PCT):
    v = vol.astype(np.float32)
    lo, hi = np.percentile(v, norm_pct)
    return np.clip((v - lo) / (hi - lo + 1e-6), 0, 1).astype(np.float32)

def _stamp_gaussian(hm, center, sigma, radius):
    shape = hm.shape; slices, locals_ = [], []
    for i in range(3):
        ci = int(round(center[i]))
        lo = max(0, ci - radius[i]); hi = min(shape[i], ci + radius[i] + 1)
        if lo >= hi: return
        slices.append(slice(lo, hi)); locals_.append(np.arange(lo, hi) - ci)
    zz, yy, xx = np.meshgrid(*locals_, indexing='ij')
    g = np.exp(-(zz**2/(2*sigma[0]**2) + yy**2/(2*sigma[1]**2) + xx**2/(2*sigma[2]**2)))
    sl = tuple(slices); hm[sl] = np.maximum(hm[sl], g.astype(np.float32))

def make_targets(vol, points_zyx, sigma=SIGMA, norm_pct=NORM_PCT, fg_thr=FG_THR,
                 pos_weight=POS_WEIGHT, near_eps=0.05, dilate_iter=2):
    """Return (heatmap, weight). weight=0 in ignore regions (bright-but-unlabeled). Coords are in the
    SAME grid as `vol` (pooled), so pooled GT points must be passed in."""
    shape = vol.shape
    radius = tuple(max(1, int(round(3*s))) for s in sigma)
    hm = np.zeros(shape, np.float32)
    for p in points_zyx: _stamp_gaussian(hm, p, sigma, radius)
    v = _normalize(vol, norm_pct)
    near = hm > near_eps
    near_dil = binary_dilation(near, iterations=dilate_iter) if near.any() else near
    ignore = (v > fg_thr) & ~near_dil
    weight = np.ones(shape, np.float32)
    weight[ignore] = 0.0
    weight[near] = pos_weight
    return hm, weight

In [ ]:
# ================= DATASET (whole isotropic 64^3 volumes; no cropping) =================
class HeatmapDataset(Dataset):
    def __init__(self, samples, sigma=SIGMA, norm_pct=NORM_PCT, fg_thr=FG_THR,
                 pos_weight=POS_WEIGHT, augment=True):
        self.samples, self.sigma = samples, sigma
        self.norm_pct, self.fg_thr, self.pos_weight = norm_pct, fg_thr, pos_weight
        self.augment = augment
        self._cache = {}
        self.items = []
        for si, (_, gp) in enumerate(samples):
            for t, pts in points_by_frame(gp).items():
                self.items.append((si, t, pts))

    def _arr(self, si):
        if si not in self._cache: self._cache[si] = open_image(self.samples[si][0])
        return self._cache[si]

    def __len__(self): return len(self.items)

    def __getitem__(self, idx):
        si, t, pts = self.items[idx]
        vol = np.asarray(self._arr(si)[t]).astype(np.float32)          # (Z,256,256) original
        vol = pool_xy(vol)                                             # (Z,64,64) isotropic
        pts_iso = [(z, y / POOL_XY, x / POOL_XY) for (z, y, x) in pts] # GT points -> pooled grid
        hm, m = make_targets(vol, pts_iso, self.sigma, self.norm_pct, self.fg_thr, self.pos_weight)
        v, h = vol, hm
        if self.augment:
            for ax in range(3):
                if np.random.rand() < 0.5:
                    v, h, m = np.flip(v, ax), np.flip(h, ax), np.flip(m, ax)
            if np.random.rand() < 0.5:                                 # xy transpose (isotropic -> valid aug)
                v, h, m = v.swapaxes(1, 2), h.swapaxes(1, 2), m.swapaxes(1, 2)
            v = v * np.float32(np.random.uniform(0.9, 1.1))
        lo, hi = np.percentile(v, self.norm_pct)
        v = np.clip((v - lo) / (hi - lo + 1e-6), 0, 1).astype(np.float32)
        to_t = lambda a: torch.from_numpy(np.ascontiguousarray(a)[None])
        return to_t(v), to_t(h), to_t(m)

In [ ]:
# ================= MODEL: symmetric 3D U-Net (isotropic grid) =================
class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv3d(cin, cout, 3, padding=1, bias=False), nn.InstanceNorm3d(cout, affine=True), nn.LeakyReLU(0.01, True),
            nn.Conv3d(cout, cout, 3, padding=1, bias=False), nn.InstanceNorm3d(cout, affine=True), nn.LeakyReLU(0.01, True))
    def forward(self, x): return self.net(x)

class UNet3D(nn.Module):
    def __init__(self, in_ch=1, base=BASE, strides=STRIDES):
        super().__init__()
        n = len(strides); chs = [base*(2**i) for i in range(n+1)]
        self.enc, self.downs = nn.ModuleList(), nn.ModuleList()
        cin = in_ch
        for i in range(n):
            self.enc.append(ConvBlock(cin, chs[i]))
            self.downs.append(nn.Conv3d(chs[i], chs[i], kernel_size=strides[i], stride=strides[i]))
            cin = chs[i]
        self.bottleneck = ConvBlock(chs[n-1], chs[n])
        self.ups, self.dec = nn.ModuleList(), nn.ModuleList()
        for i in reversed(range(n)):
            self.ups.append(nn.ConvTranspose3d(chs[i+1], chs[i], kernel_size=strides[i], stride=strides[i]))
            self.dec.append(ConvBlock(chs[i]*2, chs[i]))
        self.head = nn.Conv3d(chs[0], 1, 1)
    def forward(self, x):
        skips = []
        for enc, down in zip(self.enc, self.downs):
            x = enc(x); skips.append(x); x = down(x)
        x = self.bottleneck(x)
        for up, dec, skip in zip(self.ups, self.dec, reversed(skips)):
            x = up(x)
            if x.shape[2:] != skip.shape[2:]:
                x = F.interpolate(x, size=skip.shape[2:], mode='trilinear', align_corners=False)
            x = dec(torch.cat([x, skip], dim=1))
        return self.head(x)   # raw logits; sigmoid in loss/inference

In [ ]:
# ================= LOSS / LINK / SCORE / INFER (downstream = ORIGINAL voxel space, v1-identical) =================
def masked_mse(logits, target, weight):
    pred = torch.sigmoid(logits)
    se = (pred - target)**2 * weight
    return se.sum() / (weight.sum() + 1e-6)

def build_graph(coords_list, gate=GATE_UM):
    """NN Hungarian linker in ORIGINAL um space (same as v1 -> eval is comparable to v1's 0.808)."""
    nodes, edges, nid = [], [], 0
    prev_ids, prev_xyz = None, None
    for t, coords in enumerate(coords_list):
        coords = np.asarray(coords, float).reshape(-1, 3)
        ids = list(range(nid, nid+len(coords))); nid += len(coords)
        for i, c in zip(ids, coords): nodes.append((i, t, float(c[0]), float(c[1]), float(c[2])))
        if prev_xyz is not None and len(prev_xyz) and len(coords):
            D = cdist(prev_xyz*VOXEL, coords*VOXEL)
            r, c = linear_sum_assignment(D)
            for i, j in zip(r, c):
                if D[i, j] <= gate: edges.append((prev_ids[i], ids[j]))
        prev_ids, prev_xyz = ids, coords
    return nodes, edges

def match_nodes_per_t(gt_nodes, pred_nodes, max_um=MATCH_UM):
    gt_by_t, pr_by_t = defaultdict(list), defaultdict(list)
    for n in gt_nodes: gt_by_t[n[1]].append(n)
    for n in pred_nodes: pr_by_t[n[1]].append(n)
    g2p = {}
    for t, G in gt_by_t.items():
        P = pr_by_t.get(t, [])
        if not P: continue
        A = np.array([[g[2],g[3],g[4]] for g in G])*VOXEL
        B = np.array([[p[2],p[3],p[4]] for p in P])*VOXEL
        D = cdist(A, B); r, c = linear_sum_assignment(D)
        for i, j in zip(r, c):
            if D[i, j] <= max_um: g2p[G[i][0]] = P[j][0]
    return g2p

def edge_jaccard(gt_nodes, gt_edges, pred_nodes, pred_edges, max_um=MATCH_UM):
    g2p = match_nodes_per_t(gt_nodes, pred_nodes, max_um)
    p2g = {v: k for k, v in g2p.items()}
    pred_set, gt_set = set(map(tuple, pred_edges)), set(map(tuple, gt_edges))
    TP = FP = 0
    for a, b in pred_edges:
        if a in p2g and b in p2g:
            TP += (p2g[a], p2g[b]) in gt_set; FP += (p2g[a], p2g[b]) not in gt_set
    FN = sum(1 for u, v in gt_edges if not (u in g2p and v in g2p and (g2p[u], g2p[v]) in pred_set))
    d = TP+FP+FN
    return dict(jaccard=TP/d if d else 0.0, TP=TP, FP=FP, FN=FN,
                matched_gt=len(g2p), n_gt_nodes=len(gt_nodes), n_pred_nodes=len(pred_nodes))

def refine_centroids(vol, coords, win=REFINE_WIN):
    """Intensity-weighted local centre-of-mass on the ORIGINAL full-res volume -> recover sub-pool xy precision."""
    if len(coords) == 0: return coords
    Z, Y, X = vol.shape
    out = coords.copy().astype(np.float64); wz, wy, wx = win
    for i, (z, y, x) in enumerate(coords):
        z, y, x = int(round(z)), int(round(y)), int(round(x))
        z0, z1 = max(0, z-wz), min(Z, z+wz+1)
        y0, y1 = max(0, y-wy), min(Y, y+wy+1)
        x0, x1 = max(0, x-wx), min(X, x+wx+1)
        patch = vol[z0:z1, y0:y1, x0:x1].astype(np.float64); s = patch.sum()
        if s <= 0: continue
        zz = np.arange(z0, z1)[:, None, None]; yy = np.arange(y0, y1)[None, :, None]; xx = np.arange(x0, x1)[None, None, :]
        out[i, 0] = (patch*zz).sum()/s; out[i, 1] = (patch*yy).sum()/s; out[i, 2] = (patch*xx).sum()/s
    return out

@torch.no_grad()
def predict_heatmap(model, pooled_vol, norm_pct=NORM_PCT):
    v = _normalize(pooled_vol, norm_pct)
    x = torch.from_numpy(np.ascontiguousarray(v)[None, None]).float().to(device)  # ensure float32
    with torch.amp.autocast(device, enabled=(device=='cuda')):
        y = torch.sigmoid(model(x))
    return y[0, 0].float().cpu().numpy()

def detect_all_iso(model, arr, threshold=HM_THR, min_distance=MIN_DIST_POOLED, refine=REFINE, win=REFINE_WIN):
    """Per frame: original vol -> pool XY -> heatmap on 64^3 -> peaks -> map xy back x POOL_XY -> refine on full-res.
    Returns per-frame (N,3) coords in ORIGINAL voxel space."""
    model.eval(); out = []
    for t in range(arr.shape[0]):
        vol = np.asarray(arr[t]).astype(np.float32)          # (Z,256,256) original
        hm = predict_heatmap(model, pool_xy(vol))            # (Z,64,64) heatmap
        pk = peak_local_max(hm, min_distance=min_distance, threshold_abs=threshold,
                            exclude_border=False, num_peaks=5000)
        if len(pk) == 0:
            out.append(np.zeros((0, 3))); continue
        coords = pk.astype(np.float64)
        coords[:, 1] *= POOL_XY; coords[:, 2] *= POOL_XY     # y,x -> ORIGINAL resolution
        if refine:
            coords = refine_centroids(vol, coords, win=win)  # sub-pool xy precision on full-res
        out.append(coords)
    return out

In [ ]:
# ================= build datasets =================
samples = build_sample_list(TRAIN_DIR)
train_s, val_s = samples[:-N_VAL], samples[-N_VAL:]
print(f'{len(samples)} samples -> train {len(train_s)}, val {len(val_s)}')

train_ds = HeatmapDataset(train_s, augment=True)
val_ds   = HeatmapDataset(val_s, augment=False)
print('train frames:', len(train_ds), '| val frames:', len(val_ds))

train_loader = DataLoader(train_ds, batch_size=BATCH,
                          sampler=RandomSampler(train_ds, replacement=True, num_samples=STEPS*BATCH),
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

v, h, m = next(iter(train_loader))
print('batch:', v.shape, '(expect [B,1,64,64,64]) | hm max', float(h.max()), '| weight>0 frac', float((m>0).float().mean()))

In [ ]:
# ================= TRAIN (AMP + resume/warm-start + cosine-LR + early-stop + time-budget safe-exit) =================
import csv

model = UNet3D().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR)
scaler = torch.amp.GradScaler(device, enabled=(device=='cuda'))
print('params (M):', sum(p.numel() for p in model.parameters())/1e6)

@torch.no_grad()
def val_loss():
    model.eval(); tot = n = 0
    for v, h, m in val_loader:
        v, h, m = v.to(device), h.to(device), m.to(device)
        with torch.amp.autocast(device, enabled=(device=='cuda')):
            tot += float(masked_mse(model(v), h, m))
        n += 1
        if n >= 60: break
    return tot / max(n, 1)

def write_history_csv(history):
    with open(HISTORY_CSV, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=['epoch', 'train', 'val', 'best', 'lr', 'sec'])
        w.writeheader(); w.writerows(history)

def save_ckpt(epoch, best, patience, history, best_state, sched):
    torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), scaler=scaler.state_dict(),
                    sched=(sched.state_dict() if sched is not None else None),
                    epoch=epoch, best=best, patience=patience, history=history,
                    best_model=best_state), CKPT_LATEST)
    write_history_csv(history)

def _extract_state(ck):
    if isinstance(ck, dict) and ('best_model' in ck or 'model' in ck):
        return ck.get('best_model') or ck['model']
    return ck

def _find_eval_weights():
    if EVAL_WEIGHTS: return EVAL_WEIGHTS
    for pat in ('unet_iso_heatmap.pt', 'unet_iso_latest.pt', '*.pt'):
        hits = sorted(glob.glob(f'/kaggle/input/**/{pat}', recursive=True))
        if hits: return hits[0]
    return None

if EVAL_ONLY:
    wpath = _find_eval_weights()
    assert wpath, 'EVAL_ONLY: no .pt found under /kaggle/input -- add your v4 weights dataset (unet_iso_*.pt).'
    state = _extract_state(torch.load(wpath, map_location=device))
    model.load_state_dict(state); model.eval()
    torch.save(state, OUT_WEIGHTS)
    print(f'EVAL_ONLY: loaded weights from {wpath} -> training SKIPPED. Run the next cell for edge-Jaccard.')
else:
    # ---- resume (full ckpt) / warm-start (best weights). v4 arch != v1 -> only load unet_iso_* (v1 weights are incompatible). ----
    start_epoch, best, patience, history, best_state, sched_state = 0, 1e9, 0, [], None, None
    resume_hits = glob.glob('/kaggle/input/**/unet_iso_latest.pt', recursive=True)
    if resume_hits:
        ck = torch.load(resume_hits[0], map_location=device)
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt']); scaler.load_state_dict(ck['scaler'])
        start_epoch = ck['epoch'] + 1; best = ck['best']; patience = ck.get('patience', 0)
        history = ck.get('history', []); best_state = ck.get('best_model', None); sched_state = ck.get('sched', None)
        if best_state is not None: torch.save(best_state, OUT_WEIGHTS)
        print(f'RESUMED (full ckpt) from {resume_hits[0]} -> start epoch {start_epoch}, best val {best:.6f}, '
              f'patience {patience}' + (' (no sched -> fast-forwarding cosine)'
                                        if (COSINE_LR and sched_state is None) else ''))
    else:
        warm = None
        hits = sorted(glob.glob('/kaggle/input/**/unet_iso_heatmap.pt', recursive=True))
        if hits: warm = hits[0]
        if warm:
            state = _extract_state(torch.load(warm, map_location=device))
            model.load_state_dict(state)
            best_state = {k: v.detach().cpu() for k, v in state.items()}
            torch.save(state, OUT_WEIGHTS)
            best = val_loss(); start_epoch = EPOCHS // 2
            print(f'WARM-START from {warm} -> fresh opt, continue from epoch {start_epoch}/{EPOCHS} '
                  f'(gentle cosine LR), seeded best val {best:.6f}.')
        else:
            print('fresh start (no unet_iso_*.pt found under /kaggle/input)')

    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=ETA_MIN) if COSINE_LR else None
    if sched is not None:
        if sched_state is not None:
            sched.load_state_dict(sched_state)
        elif start_epoch > 0:
            for _ in range(start_epoch): sched.step()
        print(f'cosine LR: T_max={EPOCHS}, eta_min={ETA_MIN:.2e}, lr now {opt.param_groups[0]["lr"]:.2e}')

    epoch_times = []
    for ep in range(start_epoch, EPOCHS):
        if epoch_times:
            avg = sum(epoch_times) / len(epoch_times)
            elapsed = time.time() - NB_START
            if elapsed + avg + RESERVE_SEC > TIME_BUDGET_SEC:
                print(f'TIME BUDGET: pausing before epoch {ep} (elapsed {elapsed/3600:.2f}h, ~{avg:.0f}s/epoch). '
                      f'Checkpoint saved -> upload unet_iso_latest.pt next run to resume.')
                break
        cur_lr = opt.param_groups[0]['lr']
        model.train(); t0 = time.time(); run = 0.0
        for i, (v, h, m) in enumerate(train_loader):
            v, h, m = v.to(device), h.to(device), m.to(device)
            with torch.amp.autocast(device, enabled=(device=='cuda')):
                loss = masked_mse(model(v), h, m)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            run += float(loss)
        vl = val_loss(); dt = time.time() - t0; epoch_times.append(dt)
        if vl < best:
            best = vl; patience = 0
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
            torch.save(best_state, OUT_WEIGHTS)
        else:
            patience += 1
        if sched is not None: sched.step()
        history.append(dict(epoch=ep, train=round(run/(i+1), 6), val=round(vl, 6), best=round(best, 6),
                            lr=round(cur_lr, 8), sec=round(dt)))
        save_ckpt(ep, best, patience, history, best_state, sched)
        rem = (TIME_BUDGET_SEC - (time.time() - NB_START)) / 3600
        print(f'epoch {ep:2d} | lr {cur_lr:.2e} | train {run/(i+1):.4f} | val {vl:.4f} | best {best:.4f} '
              f'| patience {patience}/{EARLY_STOP_PATIENCE} | {dt:.0f}s | ~{rem:.2f}h budget left')
        if patience >= EARLY_STOP_PATIENCE:
            print(f'EARLY STOP at epoch {ep} (no val improvement for {EARLY_STOP_PATIENCE} epochs).')
            break

    if best_state is not None:
        torch.save(best_state, OUT_WEIGHTS)
    write_history_csv(history)
    print(f'done. best val {best:.6f} | weights -> {OUT_WEIGHTS} | resume ckpt -> {CKPT_LATEST} | trend -> {HISTORY_CSV}')

In [ ]:
# ================= Sanity: edge-Jaccard on VAL samples vs v1 (NN linking) 0.808 =================
# Same NN linker + scorer as v1, so this number is a clean detector A/B: v4 isotropic vs v1 full-res.
model.load_state_dict(torch.load(OUT_WEIGHTS, map_location=device)); model.eval()
TP = FP = FN = 0
for zp, gp in val_s[:EVAL_N_VAL]:
    arr = open_image(zp)
    coords = detect_all_iso(model, arr, threshold=HM_THR, min_distance=MIN_DIST_POOLED)
    p_nodes, p_edges = build_graph(coords, gate=GATE_UM)
    gt_nodes, gt_edges = gt_nodes_edges(gp)
    r = edge_jaccard(gt_nodes, gt_edges, p_nodes, p_edges)
    TP += r['TP']; FP += r['FP']; FN += r['FN']
    print(f"  {os.path.basename(gp)[:-5]}: J={r['jaccard']:.3f} matched {r['matched_gt']}/{r['n_gt_nodes']} pred={r['n_pred_nodes']}")
print(f'micro edge-Jaccard: {TP/(TP+FP+FN+1e-9):.4f}  (TP={TP} FP={FP} FN={FN})   [v1 full-res NN = 0.808]')

**Read the results:** compare this eval edge-Jaccard to **v1's 0.808** (identical NN linker + scorer, so
the delta is purely the isotropic-grid detector). Also watch `matched/n_gt` (detection recall) and `pred`
node counts — especially on any `44b6_` val samples (v1's weakness / where isotropic z-context should help most).

**Single-variable discipline:** this notebook changes ONLY the grid geometry vs v1 (base16, InstanceNorm,
sigma-in-µm, cosine LR all unchanged). If it beats v1, THEN try base24 / BatchNorm as further single steps.

**Checkpointing / resume:** saves `unet_iso_latest.pt` (full: model+opt+scaler+scheduler+epoch+best+history),
`unet_iso_heatmap.pt` (best weights), `history_iso.csv` each epoch to `/kaggle/working`. Names are distinct
from v1 and the arch differs — **v1 weights are NOT loadable here** (different conv shapes). To resume: save
`/kaggle/working` as a dataset, attach it next run; the train cell auto-finds `unet_iso_latest.pt` (exact
resume) or `unet_iso_heatmap.pt` (warm-start from the cosine midpoint). Early-stop patience 8; ~9h safe-exit.

**Next after training:** if eval > 0.808, fuse this detector into `submit.ipynb` (swap the model to symmetric
STRIDES + the pool/detect front-end, keep v2 linking + divisions OFF) and **LB-test**. `HM_THR` and
`MIN_DIST_POOLED` change peak DENSITY on the pooled grid, so re-calibrate them against the LB (the density
penalty is what bit v2.5) — do a single-variable submission, don't bundle.

**Eval-only:** set `EVAL_ONLY = True`, attach a `unet_iso_*.pt` weights dataset, Run All.